<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex04-perceptron-to-mlp/Ex04_01_perceptron_and_xor_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_04 · Notebook 01 — The Perceptron, and XOR

**Deep Learning for Engineering · Aalborg University · Part 1**

> **Read this before you start.** The second half of this notebook contains a
> training loop that **never converges**. That is the intended result. It is not
> a bug in the notebook and it will not be a bug in your code. Section 4 says so
> again, at the point where it happens.

This notebook is fifteen lines of NumPy and about fifty years of history. In it
you will:

1. implement a **perceptron** — one artificial neuron — in plain NumPy,
2. train it with **Rosenblatt's learning rule** and watch it solve AND,
3. point the same code at **XOR** and watch it fail, permanently,
4. prove to yourself that no setting of the weights could have worked,
5. fix it with a **two-unit hidden layer** — nine numbers, set by hand.

No PyTorch. Notebook 02 does the same thing with PyTorch, and the comparison is
much sharper if you have done it the hard way first.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_4_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex04-perceptron-to-mlp/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import Ex_4_core as core

X_and, y_and = core.logic_dataset("AND")
X_xor, y_xor = core.logic_dataset("XOR")

print("inputs (the same for every gate):")
print(X_and)
print("AND targets:", y_and)
print("XOR targets:", y_xor)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 4.2))
core.plot_logic(X_and, y_and, ax=ax1, title="AND")
core.plot_logic(X_xor, y_xor, ax=ax2, title="XOR")
plt.tight_layout()
plt.show()

**What you should see.** Two square plots with four markers each. Filled blue
means target 1, hollow red means target 0.

Look at them for a moment before going on, because the whole notebook is in these
two pictures. In the AND plot you can draw a straight line with the single filled
marker on one side and the three hollow ones on the other. In the XOR plot the
two filled markers are diagonally opposite, and so are the two hollow ones.

Try to draw the line. That failure, made precise, is the subject of this
notebook.

---

## 1 · What a perceptron is

One neuron. It takes the input vector $\mathbf{x}$, weights it, adds a bias, and
thresholds the result:

$$ \hat{y} = \mathrm{step}\bigl(\mathbf{w}^{\top}\mathbf{x} + b\bigr),
   \qquad
   \mathrm{step}(z) = \begin{cases} 1 & z > 0 \\ 0 & z \le 0 \end{cases} $$

For two inputs that is three numbers: $w_1$, $w_2$ and $b$. The set of points
where $w_1 x_1 + w_2 x_2 + b = 0$ is a straight line, and the neuron answers 1 on
one side of it and 0 on the other. **A perceptron is a line, with a rule for
which side is which.**

Two consequences follow immediately, and they are worth stating before you write
any code.

- Whatever the perceptron can do, it does by placing a line. If no line
  separates your classes, no perceptron does.
- The step function has zero derivative everywhere it is defined, so gradient
  descent cannot train it. It needs a different rule — the one in section 2 —
  and that is one reason the field later moved to smooth activations.

### Your turn

Implement the step function and the prediction. Keep both vectorised: `predict`
takes an array of shape `(N, 2)` and returns `N` values, because section 3 will
call it on a grid of forty thousand points.

A correct answer classifies all four AND rows correctly with the weights given in
the check cell.

In [ ]:
# TODO 1 --- the perceptron's forward pass --------------------------------------
# Two `...` to replace, one per function:
#   step     ->  np.where(z > 0, 1.0, 0.0)       1.0 where z > 0, else 0.0
#   predict  ->  step(X @ w + b)                  X is (N, 2), w is (2,)
def step(z):
    return ...                                    # <- np.where(z > 0, 1.0, 0.0)


def predict(X, w, b):
    return ...                                    # <- step(X @ w + b)
# ------------------------------------------------------------------------------

In [ ]:
w_demo, b_demo = np.array([1.0, 1.0]), -1.5     # a line that computes AND
core.truth_table(predict(X_and, w_demo, b_demo), y_and,
                 "hand-chosen weights on AND:")

assert np.all(predict(X_and, w_demo, b_demo) == y_and), "predict is wrong"
print("\npredict looks correct")

**What you should see.** Four rows, `4 of 4 correct`, then
`predict looks correct`.

Those weights were chosen by hand: $x_1 + x_2 - 1.5 > 0$ is true only when both
inputs are 1. You have just solved AND without any learning at all, which is
worth remembering — for problems this small, fitting is a convenience rather
than a necessity.

---

## 2 · Rosenblatt's learning rule

Rosenblatt's contribution in 1958 was not the neuron, which was already in
McCulloch and Pitts (1943). It was a rule for finding the weights from examples.

Visit the training rows one at a time. For each, compute the prediction and the
error $e = y - \hat{y}$, which can only be $-1$, $0$ or $+1$. Then

$$ \mathbf{w} \leftarrow \mathbf{w} + \eta\, e\, \mathbf{x},
   \qquad b \leftarrow b + \eta\, e $$

with $\eta$ a small positive learning rate. In words: if the answer was right,
change nothing. If the neuron said 0 and should have said 1, push the line
towards that point; if it said 1 and should have said 0, push it away.

This is not gradient descent — there is no differentiable loss here — but it has
something better. **The perceptron convergence theorem** (Novikoff, 1962) says
that if the two classes *can* be separated by a line, this rule finds one in a
finite number of steps, from any starting point, for any positive $\eta$.

Read the condition in that sentence carefully. It is the whole of section 4.

### Your turn

Implement the learning rule. About fifteen lines:

```
for each epoch:
    for each of the four rows:
        yhat = step(x . w + b)
        e    = y - yhat
        w   += lr * e * x
        b   += lr * e
    record how many of the four rows are wrong now
return w, b, errors
```

Start from `w = np.zeros(2)` and `b = 0.0` so that your run matches the numbers
below. Record the error count **at the end of each epoch**, using the weights as
they then are — that is the honest measure of whether training has finished.

In [ ]:
# TODO 2 --- Rosenblatt's learning rule ------------------------------------------
# Three `...` to replace, all inside the loop:
#   line 1  ->  y[i] - predict(X[i], w, b)      the error e: -1, 0 or +1
#   line 2  ->  w + lr * e * X[i]               move the weights
#   line 3  ->  b + lr * e                      move the bias
def train_perceptron(X, y, epochs=40, lr=0.1):
    w = np.zeros(X.shape[1])
    b = 0.0
    errors = []
    for epoch in range(epochs):
        for i in range(len(X)):
            e = ...                               # <- y[i] - predict(X[i], w, b)
            w = ...                               # <- w + lr * e * X[i]
            b = ...                               # <- b + lr * e
        errors.append(int(np.sum(predict(X, w, b) != y)))   # misclassified rows at epoch end
    return w, b, errors
# ------------------------------------------------------------------------------

## 3 · AND: it works

Run the rule on AND.

In [ ]:
w_and, b_and, err_and = train_perceptron(X_and, y_and, epochs=40, lr=0.1)

print(f"final weights: w = {np.round(w_and, 3)},  b = {b_and:.3f}")
print(f"errors, first ten epochs: {err_and[:10]}")
print(f"epochs until zero errors : {int(np.argmin(np.array(err_and) > 0))}")
print()
core.truth_table(predict(X_and, w_and, b_and), y_and, "AND, after training:")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.2))
core.plot_logic(X_and, y_and, ax=ax1, title="AND — the line it found")
core.plot_boundary(ax1, lambda G: predict(G, w_and, b_and))
core.plot_decision_line(ax1, w_and, b_and)
core.plot_errors(err_and, ax=ax2, title="AND — errors per epoch")
plt.tight_layout()
plt.show()

**What you should see.** Errors of `[3, 2, 1, 2, 0, 0, 0, 0, 0, 0]` — four
epochs of adjustment and then zero for ever — final weights around
`w = [0.2, 0.1]`, `b = -0.2`, and `4 of 4 correct`.

In the left panel a line runs between the filled marker at (1, 1) and the other
three, with the two regions shaded. In the right panel the error curve drops to
zero and stays there. **That is what convergence looks like**, and you now have
a picture of it to compare against.

The exact weights depend on the learning rate; the line does not need to be in
any particular place, only on the right side of all four points.

---

## 4 · XOR: it does not work

> ### The warning, for the second time
>
> The next cell runs the **same function** on XOR for two hundred epochs. It will
> not converge. The error count will not reach zero. Running it for two thousand
> epochs will not help, a different learning rate will not help, and a different
> initialisation will not help.
>
> **This is the intended result of this notebook.** Nothing is broken. If you
> came here from a lecture you missed: the failure you are about to produce is
> the reason funding for neural networks collapsed after 1969, and the reason
> everything after section 5 of this notebook exists.

Run it.

In [ ]:
w_xor, b_xor, err_xor = train_perceptron(X_xor, y_xor, epochs=200, lr=0.1)

print(f"final weights: w = {np.round(w_xor, 3)},  b = {b_xor:.3f}")
print(f"errors, first fifteen epochs: {err_xor[:15]}")
print(f"errors, last five epochs    : {err_xor[-5:]}")
print(f"best epoch in the whole run : {min(err_xor)} rows wrong")
print()
core.truth_table(predict(X_xor, w_xor, b_xor), y_xor, "XOR, after 200 epochs:")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.0, 4.2))
core.plot_logic(X_xor, y_xor, ax=ax1, title="XOR — the best line it could find")
core.plot_boundary(ax1, lambda G: predict(G, w_xor, b_xor))
core.plot_decision_line(ax1, w_xor, b_xor)
core.plot_errors(err_xor, ax=ax2, title="XOR — errors per epoch, for 200 epochs")
plt.tight_layout()
plt.show()

**What you should see.** The error count sits at **2 of 4 wrong** and stays
there — a flat line across the whole two hundred epochs. The best epoch in the
entire run is still 2 rows wrong. The final weights are about `w = [-0.1, 0.0]`,
`b = 0.1`.

Compare the two error plots. AND fell to zero in four epochs. XOR is a
horizontal line at 2, for ever.

Something slightly eerie is happening in the weights, and it is worth seeing.
Print them at the end of each of the first few epochs.

In [ ]:
w, b = np.zeros(2), 0.0
print("epoch      w1      w2       b     rows wrong")
for epoch in range(8):
    for i in range(4):
        e = y_xor[i] - step(X_xor[i] @ w + b)
        w = w + 0.1 * e * X_xor[i]
        b = b + 0.1 * e
    wrong = int(np.sum(predict(X_xor, w, b) != y_xor))
    print(f"{epoch:5d}  {w[0]:+.2f}   {w[1]:+.2f}   {b:+.2f}        {wrong}")

**What you should see.** After the first epoch the weights stop changing
completely: the four within-epoch updates cancel each other out exactly, and the
algorithm returns to the same point at the end of every epoch, for ever.

The loop is not stuck because of a bug or a bad learning rate. It has found a
**cycle**, which is what the perceptron rule does when no separating line exists.
The convergence theorem promised a finite number of steps *if the classes are
linearly separable*, and it is silent otherwise. This is what its silence looks
like.

---

## 5 · Why no line could have worked

The failure is not about this algorithm. It is about the model. Suppose some
$w_1, w_2, b$ solved XOR. Then all four of these would have to hold:

$$
\begin{aligned}
(0,0) \to 0:&\quad b \le 0 \\
(0,1) \to 1:&\quad w_2 + b > 0 \\
(1,0) \to 1:&\quad w_1 + b > 0 \\
(1,1) \to 0:&\quad w_1 + w_2 + b \le 0
\end{aligned}
$$

Add the second and the third:

$$ w_1 + w_2 + 2b > 0 $$

The first says $b \le 0$, so $-b \ge 0$, and therefore

$$ w_1 + w_2 + b \;=\; \bigl(w_1 + w_2 + 2b\bigr) - b \;>\; 0 $$

But the fourth line requires $w_1 + w_2 + b \le 0$. The two cannot both hold, so
**no such three numbers exist** — not approximately, not with a better optimiser,
not ever.

That is Minsky and Papert's argument from 1969, and it is four lines of algebra.

### Your turn

Check it by brute force, which is less elegant and more convincing. Draw a large
number of random lines, score each one on the four XOR rows, and report the best
accuracy anybody achieved.

A correct answer finds a best of **3 of 4**, no matter how many lines you try.

In [ ]:
# TODO 3 --- 200 000 random lines, none of which solves XOR ----------------------------
# Two `...` to replace, one per line:
#   line 1  ->  step(X_xor @ W.T + b)                 predictions, shape (4, n)
#   line 2  ->  (pred == y_xor[:, None]).sum(axis=0)  rows correct per line, shape (n,)
rng = np.random.default_rng(0)
n = 200_000
W = rng.normal(0.0, 3.0, size=(n, 2))
b = rng.normal(0.0, 3.0, size=n)

pred = ...                                        # <- step(X_xor @ W.T + b)
correct = ...                                     # <- (pred == y_xor[:, None]).sum(axis=0)

best_correct = int(correct.max())
n_perfect = int((correct == 4).sum())
# ------------------------------------------------------------------------------

In [ ]:
print(f"best of 200 000 random lines : {best_correct} of 4 rows correct")
print(f"lines that solved it entirely: {n_perfect}")
assert best_correct == 3 and n_perfect == 0, "unexpected — check the scoring"
print("\nno line solves XOR, as the algebra said")

**What you should see.** `3 of 4`, `0` perfect lines, and the closing message.

Three of four is not a near miss. It is the ceiling. And notice that the
perceptron learning rule did not even reach the ceiling — it cycled at 2 of 4 —
because the rule is designed to find a perfect separator and has no notion of a
best compromise.

**This is where the field stopped in 1969.** Minsky and Papert's book made the
argument precise, added that the obvious fix — more layers — had no known
training algorithm, and the funding went elsewhere for fifteen years.

---

## 6 · The fix: two lines, and a decision about them

Here is the fix, and it is the most useful intuition in this course.

You cannot separate XOR with one line. You *can* separate it with two, if you
are then allowed to combine their answers:

- **$h_1$ = OR**: is at least one input on? That line has (0,0) on one side and
  the other three on the other.
- **$h_2$ = NAND**: is it *not* the case that both are on? That line has (1,1) on
  one side and the other three on the other.
- **output = AND($h_1$, $h_2$)**: at least one on, and not both on. That is
  exactly XOR.

Each of those three is a single perceptron — a line — and each is a problem you
already know how to solve. The middle two form a **hidden layer**: they take the
inputs and produce a new pair of coordinates $(h_1, h_2)$. The output unit then
draws a line in *that* space rather than in the original one.

That is what the phrase "a hidden layer bends the plane so that a straight line
suffices" means (L4.1 slide 9), and it is, almost, the whole of deep learning: a
layer transforms the space, and the next layer solves an easier problem in the
new space.

Count the parameters. Two hidden units with two inputs each: $2 \times 2 = 4$
weights and 2 biases. One output unit with two inputs: 2 weights and 1 bias.
**Nine numbers.**

### Your turn — nine numbers, by hand

Fill in the nine. You are not training anything here, and that is deliberate: in
1969 nobody knew how to train this network, and the algorithm that does it is
L6.1. Everything you need is in the three bullet points above, and you already
solved AND by hand in section 1.

- `W1` is 2 × 2. Row 0 is the weights of $h_1$; row 1 the weights of $h_2$.
- `b1` has two entries, one bias per hidden unit.
- `w2` has two entries and `b2` is a scalar, for the output unit.

Signs are the thing to think about. OR fires easily, so its bias is a small
negative number; NAND fires unless both inputs are on, so its weights are
negative and its bias is positive.

A correct answer gets `4 of 4 correct` on XOR.

In [ ]:
# TODO 4 --- nine numbers that compute XOR -------------------------------------------
# Four `...` to replace. Each hidden unit is a line you already know how to draw:
#   W1  ->  np.array([[1.0, 1.0], [-1.0, -1.0]])   row 0 is OR, row 1 is NAND
#   b1  ->  np.array([-0.5, 1.5])                  OR fires if x1 + x2 > 0.5;
#                                                  NAND fires if -(x1 + x2) > -1.5
#   w2  ->  np.array([1.0, 1.0])                   the output unit is AND of the two
#   b2  ->  -1.5                                   fires only if both are 1
W1 = ...                                          # <- np.array([[1.0, 1.0], [-1.0, -1.0]])
b1 = ...                                          # <- np.array([-0.5, 1.5])
w2 = ...                                          # <- np.array([1.0, 1.0])
b2 = ...                                          # <- -1.5
assert not any(v is ... for v in (W1, b1, w2, b2)), "TODO 4: replace the four ..."
# ------------------------------------------------------------------------------

In [ ]:
def forward_mlp(X, W1, b1, w2, b2):
    H = step(X @ W1.T + b1)      # the hidden layer: two lines, two answers
    return step(H @ w2 + b2)     # the output unit: one line, in h-space

n_params = W1.size + b1.size + w2.size + 1
print(f"parameters: {n_params}")
print()
core.truth_table(forward_mlp(X_xor, W1, b1, w2, b2), y_xor,
                 "XOR, with one hidden layer of two units:")

assert np.all(forward_mlp(X_xor, W1, b1, w2, b2) == y_xor), "not solved yet"
assert n_params == 9
print("\nsolved, with nine numbers")

**What you should see.** `parameters: 9`, `4 of 4 correct`, and
`solved, with nine numbers`.

If a row is wrong, print the hidden activations `step(X_xor @ W1.T + b1)` and
check them against the table in the next section before adjusting anything. It
is nearly always a sign.

---

## 7 · Look at what the hidden layer did

The picture is worth more than the result. Plot the four points twice: in the
original input space, and in the space of the two hidden units.

In [ ]:
H = step(X_xor @ W1.T + b1)

print("  x1 x2  ->   h1 h2   ->  y")
for i in range(4):
    print(f"   {X_xor[i,0]:.0f}  {X_xor[i,1]:.0f}   ->    {H[i,0]:.0f}  {H[i,1]:.0f}"
          f"    ->  {forward_mlp(X_xor, W1, b1, w2, b2)[i]:.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 4.4))
core.plot_logic(X_xor, y_xor, ax=ax1, title="input space — no line works")
core.plot_boundary(ax1, lambda G: forward_mlp(G, W1, b1, w2, b2))
core.plot_logic(H, y_xor, ax=ax2, title="hidden space $(h_1, h_2)$ — one line does")
core.plot_decision_line(ax2, w2, b2)
plt.tight_layout()
plt.show()

**What you should see.** In the table, the two rows with target 1 — (0,1) and
(1,0) — both map to the **same** hidden coordinates (1, 1), while (0,0) maps to
(0, 1) and (1,1) maps to (1, 0). Four distinct input points have become three
points in hidden space, and the collision is exactly the two points that share a
label.

In the right-hand plot those three points are trivially separable: a single line
puts (1,1) on one side and the other two on the other.

The left-hand plot shows the network's decision regions in the original input
space, and they are no longer a half-plane — the boundary has a corner in it,
because it is the intersection of two half-planes.

**This is the sentence to carry forward:** the hidden layer did not classify
anything. It re-described the input in coordinates where the problem was easy,
and something linear finished the job. Every deep network in this course is that
idea, applied repeatedly.

---

## 8 · Before you move on

Answer these here.

1. The perceptron rule cycled at 2 errors on XOR while the best possible line
   gets 1 error. Explain the difference in one sentence.
2. You set nine numbers by hand. How would you have found them if the problem had
   had fifty inputs rather than two, and no obvious logical decomposition?
   (Name what you would need; you do not have it yet.)
3. L4.1 makes the point that feeding a perceptron the two inputs *plus their
   product* $x_1 x_2$ solves XOR immediately, with no hidden layer. Verify that
   claim informally: what would the three weights and the bias be? What is the
   name for what you just did, and why did deep learning largely replace it?
4. The step activation has zero derivative everywhere. Say what that rules out,
   and what notebook 02 will have to use instead.

*Write your answers here.*

1.
2.
3.
4.

---

Continue with **`Ex04_02_pytorch_mlp.ipynb`**, which builds the same network in
PyTorch — and, unlike 1969, trains it.